# NER Training — BioBERT on BC5CDR

Fine-tune `dmis-lab/biobert-base-cased-v1.2` for token classification on the project's
`bc5cdr_bio.jsonl` dataset (16,423 sentences).

**Labels:** B-DISEASE · I-DISEASE · B-NUTRIENT · I-NUTRIENT · B-FOOD · I-FOOD · O  
**Note:** BC5CDR is a pharmaceutical corpus — NUTRIENT here maps to drug/chemical names, not dietary nutrients.  
FOOD and SYMPTOM entities are not present in this dataset; `predict()` returns `[]` for those keys.  
**Output:** `ner_bert/` on Google Drive (zip + download → `models/en/ner_bert/` local)  
**Eval metric:** seqeval F1 (entity-level) on 10% validation split

In [ ]:
# Install dependencies not pre-installed on Colab
!pip install -q evaluate seqeval

## Setup — Mount Drive & Paths

In [ ]:
import os

from google.colab import drive
drive.mount("/content/drive")

# Upload bc5cdr_bio.jsonl to this folder on Drive before running
DRIVE_BASE = "/content/drive/MyDrive/nutrition-rag"
DATA_PATH  = f"{DRIVE_BASE}/bc5cdr_bio.jsonl"
OUTPUT_DIR = f"{DRIVE_BASE}/ner_bert"

os.makedirs(OUTPUT_DIR, exist_ok=True)
assert os.path.exists(DATA_PATH), f"Upload bc5cdr_bio.jsonl to {DRIVE_BASE}/ first"
print(f"Data: {DATA_PATH}")
print(f"Output: {OUTPUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data: /content/drive/MyDrive/nutrition-rag/bc5cdr_bio.jsonl
Output: /content/drive/MyDrive/nutrition-rag/ner_bert


In [ ]:
import json
import warnings

import numpy as np
import evaluate
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)

warnings.filterwarnings("ignore")

MODEL_CHECKPOINT = "dmis-lab/biobert-base-cased-v1.2"
        "LABEL_LIST = [\"O\", \"B-NUTRIENT\", \"I-NUTRIENT\", \"B-DISEASE\", \"I-DISEASE\", \"B-FOOD\", \"I-FOOD\"]\n"
label2id = {l: i for i, l in enumerate(LABEL_LIST)}
id2label  = {i: l for l, i in label2id.items()}

print(f"Labels: {label2id}")

Labels: {'O': 0, 'B-NUTRIENT': 1, 'I-NUTRIENT': 2, 'B-DISEASE': 3, 'I-DISEASE': 4}


## Data Loading

In [ ]:
rows = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

# 70/30 deterministic split — bc5cdr_bio.jsonl merges all original splits
split_idx = int(len(rows) * 0.7)
train_rows, val_rows = rows[:split_idx], rows[split_idx:]

def rows_to_dataset(rows: list) -> Dataset:
    return Dataset.from_dict({
        "tokens": [r["tokens"] for r in rows],
        "labels": [[label2id[l] for l in r["labels"]] for r in rows],
    })

train_ds = rows_to_dataset(train_rows)
val_ds   = rows_to_dataset(val_rows)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

Train: 14780 | Val: 1643


## Preprocessing — Subword Alignment

BioBERT uses WordPiece. Each original token may split into multiple subword tokens.
The label for a token's first subword keeps the original label;
continuation subwords and special tokens (`[CLS]`/`[SEP]`) are masked with `-100`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_and_align(batch: dict) -> dict:
    encoding = tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=512,
        padding=False,
    )
    aligned_labels = []
    for i, label_ids in enumerate(batch["labels"]):
        word_ids = encoding.word_ids(batch_index=i)
        prev_word_id = None
        row_labels = []
        for word_id in word_ids:
            if word_id is None:
                row_labels.append(-100)
            elif word_id != prev_word_id:
                row_labels.append(label_ids[word_id])
            else:
                row_labels.append(-100)
            prev_word_id = word_id
        aligned_labels.append(row_labels)
    encoding["labels"] = aligned_labels
    return encoding

train_tok = train_ds.map(tokenize_and_align, batched=True, remove_columns=["tokens", "labels"])
val_tok   = val_ds.map(tokenize_and_align,   batched=True, remove_columns=["tokens", "labels"])

print(f"Train tokenized: {len(train_tok)} | Val tokenized: {len(val_tok)}")

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/14780 [00:00<?, ? examples/s]

Map:   0%|          | 0/1643 [00:00<?, ? examples/s]

Train tokenized: 14780 | Val tokenized: 1643


## Training

In [ ]:
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_labels, true_preds = [], []
    for pred_row, label_row in zip(predictions, labels):
        row_true, row_pred = [], []
        for p, l in zip(pred_row, label_row):
            if l != -100:
                row_true.append(id2label[l])
                row_pred.append(id2label[p])
        true_labels.append(row_true)
        true_preds.append(row_pred)

    result = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "f1":        result["overall_f1"],
        "precision": result["overall_precision"],
        "recall":    result["overall_recall"],
    }


model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,

    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f"Training on {len(train_tok)} examples | {training_args.num_train_epochs} epochs | fp16={training_args.fp16}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored 

Training on 14780 examples | 3 epochs | fp16=True


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,No log,0.057304,0.893177,0.902919,0.883644
2,0.202051,0.063152,0.890204,0.876457,0.904390
3,0.035285,0.069268,0.893067,0.893470,0.892664


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1386, training_loss=0.0919506491348685, metrics={'train_runtime': 95.4294, 'train_samples_per_second': 464.637, 'train_steps_per_second': 14.524, 'total_flos': 2158330962829080.0, 'train_loss': 0.0919506491348685, 'epoch': 3.0})

## Evaluation

In [ ]:
metrics = trainer.evaluate()

print(f"\nValidation results (best checkpoint):")
print(f"  F1:        {metrics['eval_f1']:.4f}")
print(f"  Precision: {metrics['eval_precision']:.4f}")
print(f"  Recall:    {metrics['eval_recall']:.4f}")


Validation results (best checkpoint):
  F1:        0.8938
  Precision: 0.9033
  Recall:    0.8845


## Save Model

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Saved → {OUTPUT_DIR}")
print("Files:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1e6
    print(f"  {fname:<40} {size_mb:.1f} MB")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved → /content/drive/MyDrive/nutrition-rag/ner_bert
Files:
  checkpoint-1386                          0.0 MB
  checkpoint-462                           0.0 MB
  checkpoint-924                           0.0 MB
  config.json                              0.0 MB
  model.safetensors                        430.9 MB
  tokenizer.json                           0.7 MB
  tokenizer_config.json                    0.0 MB
  training_args.bin                        0.0 MB


## Download — Zip & Export to Local

Zip chỉ lấy files cần thiết cho inference (bỏ optimizer states).
Giải nén vào `D:\FoodRecomendationSystem\models\en\ner_bert\`

In [ ]:
import shutil
from google.colab import files

INFERENCE_FILES = [
    "config.json",
    "model.safetensors",
    "tokenizer_config.json",
    "tokenizer.json",
    "vocab.txt",
    "special_tokens_map.json",
]

# Copy only inference files to a temp dir before zipping (skip optimizer states)
EXPORT_DIR = "/content/ner_bert_export"
os.makedirs(EXPORT_DIR, exist_ok=True)
for fname in INFERENCE_FILES:
    src = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, EXPORT_DIR)

shutil.make_archive("/content/ner_bert", "zip", EXPORT_DIR)

zip_size = os.path.getsize("/content/ner_bert.zip") / 1e6
print(f"Zip size: {zip_size:.1f} MB")
print("Extracting to: D:\\FoodRecomendationSystem\\models\\en\\ner_bert\\")

files.download("/content/ner_bert.zip")

Zip size: 400.2 MB
Extracting to: D:\FoodRecomendationSystem\models\en\ner_bert\


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>